# Regression

## Imports

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display

## Load Datasets

In [3]:
base_dir = os.path.join(os.getcwd(), '..')
processed_data_dir = os.path.join(base_dir, 'processed_data')

open_meteo_filepath = os.path.join(processed_data_dir, 'open_meteo_processed_data.csv')
open_meteo_df = pd.read_csv(open_meteo_filepath, parse_dates=['datetime'])

ausgrid_filepath = os.path.join(processed_data_dir, 'ausgrid_processed_data.csv')
ausgrid_df = pd.read_csv(ausgrid_filepath, parse_dates=['datetime'])

### Settings

In [12]:
customer_testing_split = 0.15
datetime_testing_split = 0.15

First, we merge the smart-meter and weather data on `datetime`. A left join keeps every smart-meter row, even if there is no matching weather record. We put the merged data in time order.

In [13]:
# Merge weather and smart-meter data
data = ausgrid_df.merge(open_meteo_df, on='datetime', how='left').sort_values('datetime')

We create two separate test sets:
- Customer test: How well does the model generalize to customers it has never seen?
- Datetime test: How well does it predict a later period for customer whose earlier data is in training?

First, we choose customers to hold out. Next, we choose a later-date test period for the remaining customers. We report the actual propotions of each dataset as percentages of the original dataset. The actual proportions may differ slightly from the requested splits because customers and timestamps are kept together. Furthermore, the code chooses the latest segment by row count, not necessarily a full year, so the resulting date range may not cover all seasons.

In [14]:
rng = np.random.default_rng(42)

# Hold out whole customers, choosing the random prefix closest to the target row count.
customer_counts = data.groupby("Customer", sort=False).size()
shuffled_customers = rng.permutation(customer_counts.index.to_numpy())
customer_counts_shuffled = customer_counts.loc[shuffled_customers]
cumulative_customer_rows = customer_counts_shuffled.cumsum()

customer_target_rows = customer_testing_split * len(data)
customer_count_end = (cumulative_customer_rows - customer_target_rows).abs().argmin() + 1
customer_test_customers = shuffled_customers[:customer_count_end]

customer_test_mask = data["Customer"].isin(customer_test_customers)
customer_test_data = data.loc[customer_test_mask].copy()
remaining_data = data.loc[~customer_test_mask]

# Hold out the latest contiguous datetime segment, closest to the target row count.
rows_per_datetime = remaining_data.groupby("datetime", sort=True).size()
cumulative_tail_rows = rows_per_datetime.iloc[::-1].cumsum()

datetime_target_rows = (datetime_testing_split/(1 - customer_testing_split)) * len(remaining_data)
datetime_start = (cumulative_tail_rows.sub(datetime_target_rows).abs().idxmin())

datetime_test_mask = remaining_data["datetime"] >= datetime_start
datetime_test_data = remaining_data.loc[datetime_test_mask].copy()
train_data = remaining_data.loc[~datetime_test_mask].copy()

print(f"Train: {len(train_data) / len(data):.1%} of all rows")
print(f"Customer test: {len(customer_test_data) / len(data):.1%} of all rows")
print(f"Datetime test: {len(datetime_test_data) / len(data):.1%} of all rows")

datetime_range = datetime_test_data["datetime"].agg(["min", "max"])
print(f"Datetime test range: {datetime_range['min']} to {datetime_range['max']}")

Train: 70.0% of all rows
Customer test: 15.0% of all rows
Datetime test: 15.0% of all rows
Datetime test range: 2012-12-19 14:30:00 to 2013-06-30 23:30:00


## Average Baseline

The goal of this baseline is to predict PV generation (`Pgen`) from historical averages at the time time-of-year and time-of-day. `get_slots` converts each timestamp into a month slot (0-12) and half-hour slot (0-47).  

First we will train the model to predict power generation for unseen customers. The `monthly_profile` averages `Pgen` in the training data for each month and half-hour slot. Observations from different years, days, and customers with the same month and slot are averaged together. `Pgen` is then predicted for unseen customers (`customer_test_data`). 

In [21]:
def get_slots(data):
    months = data["datetime"].dt.month
    slots = data["datetime"].dt.hour * 2 + data["datetime"].dt.minute // 30
    return months, slots

train_months, train_slots = get_slots(train_data)
customer_test_months, customer_test_slots = get_slots(customer_test_data)

# Average historical PV generation, keeping month and half-hour slot distinct
monthly_profile = (
    pd.DataFrame({
        "month": train_months.to_numpy(),
        "slot": train_slots.to_numpy(),
        "Pgen": train_data["Pgen"].to_numpy(),
    })
    .groupby(["month", "slot"])["Pgen"]
    .mean())

test_keys = pd.MultiIndex.from_arrays([
    customer_test_months.to_numpy(),
    customer_test_slots.to_numpy(),
])

customer_test_data["Pgen_Pred_avg-baseline"] = monthly_profile.reindex(test_keys).to_numpy()

Next, we predict `Pgen` for the later-date test set, using only the training data. In this part, `monthly_profile` averages training `Pgen` separately for each customer, month, and half-hour slot. `Pgen` is then predicted for the unseen days ('datetime_test_data`).

In [22]:
datetime_test_months, datetime_test_slots = get_slots(datetime_test_data)

monthly_profile = (
    pd.DataFrame({
        "Customer": train_data["Customer"].to_numpy(),
        "month": train_months.to_numpy(),
        "slot": train_slots.to_numpy(),
        "Pgen": train_data["Pgen"].to_numpy(),
    })
    .groupby(["Customer", "month", "slot"])["Pgen"]
    .mean())

test_keys = pd.MultiIndex.from_arrays([
    datetime_test_data["Customer"].to_numpy(),
    datetime_test_months.to_numpy(),
    datetime_test_slots.to_numpy(),
])

datetime_test_data["Pgen_Pred_avg-baseline"] = monthly_profile.reindex(test_keys).to_numpy()